In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
# 1. CẤU HÌNH MÔI TRƯỜNG
MY_JAVA_HOME = r"D:\java\openjdk-17.0.18b8"
MY_HADOOP_HOME = r"D:\java\hadoop-3.4.3"
os.environ["JAVA_HOME"] = MY_JAVA_HOME
os.environ["HADOOP_HOME"] = MY_HADOOP_HOME
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(MY_HADOOP_HOME, "bin"))
# 2. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName('MetroPT3_SQL_Analysis_Group16') \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://10.125.222.18:9000") \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print("-> Trạng thái: Spark Session cho SQL đã sẵn sàng!")
# 3. ĐỌC DỮ LIỆU SẠCH VÀ TẠO ĐẶC TRƯNG THỜI GIAN
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
print("-> Đang nạp dữ liệu Parquet từ HDFS...")
df = spark.read.parquet(HDFS_CLEAN_FOR_SQL)
# Cột 'timestamp' đã là dạng thời gian chuẩn, chỉ cần trích xuất thẳng ra giờ, tháng, ngày
df = df.withColumn('hour',    F.hour('timestamp')) \
       .withColumn('month',   F.month('timestamp')) \
       .withColumn('weekday', F.dayofweek('timestamp')) \
       .withColumn('date',    F.to_date('timestamp'))
# 4. LƯU TRỮ VÀO BỘ NHỚ ĐỆM (CACHE) & TẠO VIEW TRUY VẤN
print("-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...")
df.persist(StorageLevel.MEMORY_AND_DISK)
# Kích hoạt hành động (Trigger) để ép Spark thực sự nạp dữ liệu vào RAM
total_rows = df.count()
print(f"-> Cached thành công: {total_rows:,} rows")
# Tạo bảng ảo tên là 'sensor' để sử dụng cú pháp spark.sql()
df.createOrReplaceTempView('sensor')
print("=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===")

-> Trạng thái: Spark Session cho SQL đã sẵn sàng!
-> Đang nạp dữ liệu Parquet từ HDFS...
-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...
-> Cached thành công: 1,516,948 rows
=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===


In [2]:
# Q10: TOP 10 NGÀY CÓ TẦN SUẤT ĐÓNG/NGẮT CAO NHẤT
q10 = spark.sql('''
WITH state_changes AS (
    SELECT
        date,
        timestamp,
        COMP,
        LAG(COMP,1,COMP) OVER(ORDER BY timestamp) AS prev_comp
    FROM sensor
)
SELECT
    date,
    COUNT(*) AS so_lan_dong_ngat
FROM state_changes
WHERE COMP <> prev_comp
GROUP BY date
ORDER BY so_lan_dong_ngat DESC
LIMIT 10
''')
print("\n========== EXECUTION PLAN ==========")
q10.explain(True)
print("\n--- TOP 10 NGÀY CÓ TẦN SUẤT ĐÓNG/NGẮT CAO NHẤT ---")
df_q10 = q10.toPandas()
try:
    display(df_q10)
except NameError:
    print(df_q10.to_string(index=False))


========== EXECUTION PLAN ==========
== Parsed Logical Plan ==
CTE [state_changes]
:  +- 'SubqueryAlias state_changes
:     +- 'Project ['date, 'timestamp, 'COMP, 'LAG('COMP, 1, 'COMP) windowspecdefinition('timestamp ASC NULLS FIRST, unspecifiedframe$()) AS prev_comp#712]
:        +- 'UnresolvedRelation [sensor], [], false
+- 'GlobalLimit 10
   +- 'LocalLimit 10
      +- 'Sort ['so_lan_dong_ngat DESC NULLS LAST], true
         +- 'Aggregate ['date], ['date, 'COUNT(1) AS so_lan_dong_ngat#711]
            +- 'Filter NOT ('COMP = 'prev_comp)
               +- 'UnresolvedRelation [state_changes], [], false

== Analyzed Logical Plan ==
date: date, so_lan_dong_ngat: bigint
WithCTE
:- CTERelationDef 0, false
:  +- SubqueryAlias state_changes
:     +- Project [date#24, timestamp#1, COMP#9, prev_comp#712]
:        +- Project [date#24, timestamp#1, COMP#9, prev_comp#712, prev_comp#712]
:           +- Window [lag(COMP#9, -1, COMP#9) windowspecdefinition(timestamp#1 ASC NULLS FIRST, specifiedwind

,date,so_lan_dong_ngat
0,2020-05-27,2814
1,2020-04-20,2012
2,2020-05-26,1612
3,2020-06-12,1586
4,2020-04-17,1528
5,2020-07-21,976
6,2020-07-22,762
7,2020-05-28,528
8,2020-07-15,256
9,2020-04-21,208
